# Exploding Kittens — Hand-Test

Manuelles Durchspielen einer generierten Implementierung **ohne** Generation, Judge oder Check-Pipeline.

**Ziel:** Regelwerk (`inputs/game_rules.pdf`) neben dem Code halten und Kernmechaniken selbst verifizieren.

**Ablauf:**
1. `BACKEND` wählen (`gpt` / `codex` / `claude` / `glm`) und Spielerzahl setzen
2. Setup-Zellen ausführen → `new_game()`
3. Zwei Menschen abwechselnd: `make_turn_ui(game, state)` (Textfeld + live Refresh)
4. Checkliste unten abhaken; Notizen in `meeting/10.7/MANUAL_TESTS.txt`

Conda: `boardbench` · **Kernel neu starten** nach Backend-Wechsel.

In [2]:
import importlib.util
import sys
from pathlib import Path

REPO_ROOT = Path(".").resolve()
OUTPUT_DIR = REPO_ROOT / "outputs"
RULES_PATH = REPO_ROOT / "inputs" / "game_rules.pdf"

# --- Konfiguration ---------------------------------------------------------
BACKEND = "gpt"  # gpt | codex | claude | glm  (expl_<backend>_ag.py)
NUM_PLAYERS = 4
START_PLAYER = 0
SEED = 42

# Zwei Menschen am gleichen Rechner: Indices der menschlichen Spieler.
# Leer = alle Spieler sind menschlich (jeder Zug wartet auf input).
HUMAN_PLAYERS = {0, 1}

AUTO_VIEW = True   # information_state folgt current_player
SHOW_CHEAT = False # True = immer full render (debug)

BACKENDS = {
    "gpt": "expl_gpt_ag.py",
    "codex": "expl_codex_ag.py",
    "claude": "expl_claude_ag.py",
    "glm": "expl_glm_ag.py",
}

if BACKEND not in BACKENDS:
    raise ValueError(f"BACKEND must be one of {sorted(BACKENDS)}")
CODE_PATH = OUTPUT_DIR / BACKENDS[BACKEND]

if not CODE_PATH.exists():
    raise FileNotFoundError(f"Missing {CODE_PATH} — restore from git history or run generation")
if not RULES_PATH.exists():
    print(f"WARN: rulebook not at {RULES_PATH} — run: python generation/activate_game.py exploding_kittens")

humans = sorted(HUMAN_PLAYERS) if HUMAN_PLAYERS else list(range(NUM_PLAYERS))
print(f"backend={BACKEND}  code={CODE_PATH.name}  players={NUM_PLAYERS}  humans={humans}")

backend=gpt  code=expl_gpt_ag.py  players=4  humans=[0, 1]


In [3]:
from IPython.display import clear_output, display

TERMINAL = -1
CHANCE = -2
SIMULTANEOUS = -3
PLAYER_LABELS = {TERMINAL: "TERMINAL", CHANCE: "CHANCE", SIMULTANEOUS: "SIMULTANEOUS"}


def load_game_module(code_path: Path):
    module_name = f"manual_test_{code_path.stem}"
    spec = importlib.util.spec_from_file_location(module_name, code_path)
    if spec is None or spec.loader is None:
        raise RuntimeError(f"could not import {code_path}")
    module = importlib.util.module_from_spec(spec)
    sys.modules[module_name] = module
    spec.loader.exec_module(module)
    return module


def make_game(Game, *, num_players: int, start_player: int = 0, seed: int = 42):
    """Construct Game across backends with different __init__ signatures."""
    for kwargs in (
        {"num_players": num_players, "start_player": start_player},
        {"num_players": num_players, "seed": seed},
        {"num_players": num_players},
    ):
        try:
            return Game(**kwargs)
        except TypeError:
            continue
    return Game(num_players)


def player_label(game, state) -> str:
    cp = game.current_player(state)
    return PLAYER_LABELS.get(cp, f"p{cp}")


def view_for_state(game, state, *, manual_view: int | None = None) -> int:
    if manual_view is not None:
        return manual_view
    if AUTO_VIEW:
        cp = game.current_player(state)
        if cp >= 0:
            return cp
    return 0


def turn_banner(game, state) -> None:
    cp = game.current_player(state)
    if cp == CHANCE:
        print(">>> CHANCE / Setup — Zufallszug (Index oder Action-String)")
    elif cp == TERMINAL:
        print(">>> Spiel beendet")
    elif cp in HUMAN_PLAYERS or not HUMAN_PLAYERS:
        print(f">>> Spieler {cp} am Zug ({player_label(game, state)})")
    else:
        print(f">>> Bot-Spieler {cp} — trotzdem input für Testzug möglich")


def show(game, state, *, view_player: int | None = None, full: bool | None = None) -> None:
    vp = view_for_state(game, state, manual_view=view_player)
    cheat = SHOW_CHEAT if full is None else full
    print("--- status ---")
    print(f"current={player_label(game, state)}  terminal={game.is_terminal(state)}")
    if game.is_terminal(state):
        print("returns:", game.returns(state))
    print()
    turn_banner(game, state)
    print()
    print(f"--- Sicht Spieler {vp} ---")
    print(game.information_state(state, vp))
    if cheat:
        print()
        print("--- full debug (cheat sheet) ---")
        print(game.render(state))
    actions = game.legal_actions(state)
    print()
    print(f"--- legal actions ({len(actions)}) ---")
    for i, action in enumerate(actions):
        name = game.action_to_name(action)
        print(f"  [{i:3d}] {action}  (name={name!r})")


def play(game, state, action: str):
    legal = game.legal_actions(state)
    if action not in legal:
        resolved = game.name_to_action(action)
        if resolved not in legal and action not in legal:
            raise ValueError(f"Illegal action {action!r}; legal={legal[:8]}{'...' if len(legal) > 8 else ''}")
        action = resolved if resolved in legal else action
    return game.apply_action(state, action)


def play_index(game, state, index: int):
    legal = game.legal_actions(state)
    if not (0 <= index < len(legal)):
        raise IndexError(f"index {index} out of range 0..{len(legal) - 1}")
    return play(game, state, legal[index])


def play_session(game, state, *, manual_view: int | None = None):
    """REPL: Zahl = Action-Index, sonst Action-String. q/debug/view N."""
    fixed_view = manual_view
    while not game.is_terminal(state):
        clear_output(wait=True)
        show(game, state, view_player=fixed_view, full=False)
        raw = input("Zug (Index/Action, q=quit): ").strip()
        if raw.lower() in {"q", "quit", "exit"}:
            print("gestoppt")
            return state
        if raw.lower() == "debug":
            show(game, state, view_player=fixed_view, full=True)
            continue
        if raw.lower().startswith("view "):
            fixed_view = int(raw.split()[1])
            continue
        try:
            if raw.isdigit():
                state = play_index(game, state, int(raw))
            else:
                state = play(game, state, raw)
        except Exception as exc:
            print(f"FEHLER: {exc}")
            input("Enter zum Fortfahren...")
        print()
    clear_output(wait=True)
    show(game, state, view_player=fixed_view, full=True)
    return state


def make_turn_ui(game, state, *, manual_view: int | None = None):
    """Textfeld + Button: Zug eingeben, sofort anwenden und neu anzeigen."""
    try:
        import ipywidgets as widgets
    except ImportError:
        print("ipywidgets fehlt — Fallback: play_session(game, state)")
        print("Install: conda install -n boardbench ipywidgets")
        return None

    holder = {"state": state, "fixed_view": manual_view}

    move_box = widgets.Text(
        description="Zug:",
        placeholder="Index (z.B. 0) oder Action-String",
        continuous_update=False,
    )
    go_btn = widgets.Button(description="Spielen", button_style="primary")
    reset_btn = widgets.Button(description="Neu")
    out = widgets.Output()

    def refresh() -> None:
        with out:
            clear_output(wait=True)
            full = game.is_terminal(holder["state"])
            show(game, holder["state"], view_player=holder["fixed_view"], full=full)

    def submit(_=None) -> None:
        raw = move_box.value.strip()
        move_box.value = ""
        if not raw:
            refresh()
            return
        if raw.lower() in {"q", "quit", "exit"}:
            with out:
                clear_output(wait=True)
                print("gestoppt")
            return
        if raw.lower() == "debug":
            with out:
                clear_output(wait=True)
                show(game, holder["state"], view_player=holder["fixed_view"], full=True)
            return
        if raw.lower().startswith("view "):
            holder["fixed_view"] = int(raw.split()[1])
            refresh()
            return
        try:
            if raw.isdigit():
                holder["state"] = play_index(game, holder["state"], int(raw))
            else:
                holder["state"] = play(game, holder["state"], raw)
        except Exception as exc:
            with out:
                clear_output(wait=True)
                print(f"FEHLER: {exc}")
                show(game, holder["state"], view_player=holder["fixed_view"], full=False)
            return
        refresh()

    def reset(_=None) -> None:
        holder["state"] = game.initial_state()
        holder["fixed_view"] = manual_view
        refresh()

    go_btn.on_click(submit)
    reset_btn.on_click(reset)
    move_box.on_submit(submit)

    ui = widgets.VBox([
        widgets.HBox([move_box, go_btn, reset_btn]),
        widgets.HTML("<small>Enter oder «Spielen» — Index aus legal actions oder Action-String</small>"),
        out,
    ])
    refresh()
    display(ui)
    return holder


def deck_len(state) -> int | None:
    if hasattr(state, "draw_pile"):
        return len(state.draw_pile)
    if hasattr(state, "deck"):
        return len(state.deck)
    return None


def verify_setup(game, state) -> None:
    """Quick sanity checks (schema-tolerant across backends)."""
    n = state.num_players
    hand_lens = [len(h) for h in state.hands]
    draw_n = deck_len(state)
    print(f"players={n}")
    print(f"hand sizes={hand_lens} (expect 8 each at start)")
    if draw_n is not None:
        print(f"draw pile={draw_n} (expect 51-7n={51 - 7 * n})")
    print(f"start legal actions={len(game.legal_actions(state))}")

In [4]:
module = load_game_module(CODE_PATH)
Game = module.Game
game = make_game(Game, num_players=NUM_PLAYERS, start_player=START_PLAYER, seed=SEED)
state = game.initial_state()

show(game, state)
print()
verify_setup(game, state)

--- status ---
current=p0  terminal=False

>>> Spieler 0 am Zug (p0)

--- Sicht Spieler 0 ---
you=p0
phase=turn current=0 turns=1
alive=p0:alive,p1:alive,p2:alive,p3:alive
your_hand=[entschaerfung,angriff,hops,wunsch,blick_in_die_zukunft,noe,katzenkarte_3,katzenkarte_4]
hand_sizes=p0:8,p1:8,p2:8,p3:8
draw_count=23
discard=[]

--- legal actions (287) ---
  [  0] pass  (name='pass')
  [  1] play:hops  (name='move:hand_hops->discard')
  [  2] play:angriff  (name='move:hand_angriff->discard')
  [  3] play:blick_in_die_zukunft  (name='move:hand_blick_in_die_zukunft->discard')
  [  4] play:wunsch->p1  (name='move:hand_wunsch->discard_for_p1')
  [  5] play:wunsch->p2  (name='move:hand_wunsch->discard_for_p2')
  [  6] play:wunsch->p3  (name='move:hand_wunsch->discard_for_p3')
  [  7] five:entschaerfung+angriff+hops+wunsch+blick_in_die_zukunft->discard:entschaerfung  (name='move:five_entschaerfung_and_angriff_and_hops_and_wunsch_and_blick_in_die_zukunft->discard_take_entschaerfung')
  [  8] fiv

## Einzelzüge

```python
state = play(game, state, "pass")          # oder play_index(game, state, 0)
show(game, state)
```

Nach Modulwechsel: Kernel neu starten und Setup-Zellen erneut ausführen.

In [ ]:
# Beispiel: ein Zug (anpassen oder Zelle leer lassen)
# state = play(game, state, "pass")
# show(game, state)

## Interaktive Session (zwei Spieler)

**Empfohlen:** `make_turn_ui(game, state)` — Textfeld oben, Zug-Index eingeben, Enter → sofort neuer Stand.

Alternativ `play_session()` (Terminal-Input, refresht mit `clear_output`).

- Zahl = Action-Index aus «legal actions», sonst Action-String
- `debug`, `view 2`, `q` auch im Textfeld
- Perspektive folgt `current_player` (`AUTO_VIEW`)

In [ ]:
state = game.initial_state()
turn_ui = make_turn_ui(game, state)

# Fallback ohne ipywidgets:
# state = play_session(game, state)

## Szenario-Checkliste (Regelwerk + Code)

Aus Judge-Reviews abgeleitet — manuell anstreben oder mit `show()` nach jedem Schritt prüfen.

| # | Szenario | Erwartung | OK? | Notiz |
|---|----------|-----------|-----|-------|
| 1 | Setup 2/4/5 Spieler | 8 Karten/Hand, 1 Defuse/Hand, Deckgröße 51−7n | | |
| 2 | Zug: Karten spielen dann ziehen | Phase `turn`, am Ende Draw | | |
| 3 | Angriff | Nächster Spieler 2 Züge | | |
| 4 | Hops unter Angriff | turns_remaining sinkt, kein vorzeitiger Wechsel | | |
| 5 | Nö! / Doch | Parität entscheidet Cancel vs. Resolve | | |
| 6 | Blick in die Zukunft | Top-3 sichtbar, danach Draw | | |
| 7 | Wunsch | Zielspieler gibt Karte | | |
| 8 | 2er/3er/5er-Kombos | Pair steal, Triple named card, Five from discard | | |
| 9 | Defuse + Einfügen | EK zurück ins Deck, Zug endet | | |
| 10 | EK ohne Defuse | Elimination, terminal bei 1 Überlebenden | | |
| 11 | 2-Spieler-Variante | nur 2 Defuse im Nachziehstapel? EK-Anzahl? | | |
| 12 | information_state | fremde Hände + Deckreihenfolge versteckt | | |

Ergebnisse: `meeting/10.7/MANUAL_TESTS.txt`

## Varianten vergleichen (optional)

Nur Startzustand — zeigt z. B. 287 vs. 6 legale Startaktionen (Five-Combo-Enumeration).

In [ ]:
COMPARE_PATHS = [OUTPUT_DIR / name for name in BACKENDS.values()]

for path in COMPARE_PATHS:
    if not path.exists():
        print(f"skip {path.name} (missing)")
        continue
    mod = load_game_module(path)
    g = make_game(mod.Game, num_players=NUM_PLAYERS, start_player=START_PLAYER, seed=SEED)
    s = g.initial_state()
    phase = getattr(s, "phase", "?")
    print(f"{path.name:22s}  legal_start={len(g.legal_actions(s)):4d}  phase={phase}")